# Valid data in Bronze zone

In [2]:
from pyspark.sql import SparkSession, DataFrame
import os
import pyspark.sql.functions as F
from dotenv import load_dotenv

load_dotenv('../../../../.env_file/infra.env'), load_dotenv('../../../../.env_file/app.env')

(True, True)

In [3]:
spark = (
    SparkSession.builder
    .appName('ingest data to bronze zone')
    # jar files supporting read/write postgres, S3, hive (data lake)
    .config(
        'spark.jars',
        """
            ../../../../spark/jar/postgresql-42.7.11.jar,
            ../../../../spark/jar/hadoop-aws-3.4.2.jar,
            ../../../../spark/jar/bundle-2.29.52.jar
        """
    )
    .config("spark.hadoop.fs.s3a.access.key", os.getenv('AWS_ACCESS_KEY_ID'))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv('AWS_SECRET_ACCESS_KEY'))
    # Hive metastore
    .config("spark.hadoop.hive.metastore.uris", "thrift://localhost:9083")
    .config("spark.sql.catalogImplementation", "hive")
    # create schema using spark (instead of using trino)
    .config("spark.sql.warehouse.dir", "s3a://flezi-data/warehouse/")
    .enableHiveSupport()
    .getOrCreate()
)

spark

26/06/25 03:53:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [3]:
df_origin = (
    spark.read.format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option("url", f"jdbc:postgresql://localhost:{os.getenv('COM_DB_PORT')}/{os.getenv('COM_DB_NAME')}")
    .option("dbtable", "public.nyc_taxi")
    .option("user", os.getenv("COM_DB_USER"))
    .option("password", os.getenv("COM_DB_PASSWORD"))
    .load()
)

In [4]:
df_bronze = spark.read.table('bronze.nyc_taxi')

In [5]:
# 2009-02-08 00:38:49
# 2009-01-31 23:59:55

df_bronze.filter(
    (df_bronze.pickup_datetime <= '2009-02-08 00:38:49') &
    (df_bronze.pickup_datetime > '2009-01-31 23:59:55')
).count()

17000

In [6]:
df_origin.filter(
    (df_origin.pickup_datetime <= '2009-02-08 00:38:49') &
    (df_origin.pickup_datetime > '2009-01-31 23:59:55')
).count()

34000

In [3]:
# Connection variables
trino_url = "jdbc:trino://localhost:8082"
trino_user = "trino"
trino_password = ""
target_table = "datalake.taxi_data.nyc_taxi"

# Read table into a DataFrame
df_bronze = (
    spark.read
    .option("pushDownPredicate", "false")
    .format("jdbc")
    .option("url", trino_url)
    .option("driver", "io.trino.jdbc.TrinoDriver")
    .option("dbtable", target_table)
    .option("user", trino_user)
    .option("password", trino_password)
    .load()
)
df_bronze.count()

7781765

In [ ]:
def validate_bronze(df_bronze:DataFrame, df_origin:DataFrame, valid_date:str):
    count_bronze = df_bronze.filter(
        df_bronze['pickup_datetime'] < valid_date
    ).count()

    count_origin = df_origin.filter(
        df_origin['pickup_datetime'] < valid_date
    ).count()

    print(count_bronze, count_origin)
    return count_bronze == count_origin

In [7]:
valid_date = '2009-02-02'
validate_bronze(df_bronze, df_origin, valid_date)

144619 144619


True

In [8]:
valid_date = '2009-03-01'
validate_bronze(df_bronze, df_origin, valid_date)

280322 280325


False

In [5]:
df = spark.read.table('bronze.nyc_taxi')
df.count()

15417347

In [6]:
df.createOrReplaceTempView('nyc_taxi')
spark.sql('''
    delete from nyc_taxi
    where month(pickup_datetime)=6 and year(pickup_datetime)=2019;
''').filter(
    (F.month(F.col('pickup_datetime'))==6) &
    (F.year(F.col('pickup_datetime'))==2019)
).show()

26/06/25 03:57:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


AnalysisException: [UNSUPPORTED_FEATURE.TABLE_OPERATION] The feature is not supported: Table `spark_catalog`.`bronze`.`nyc_taxi` does not support DELETE. Please check the current catalog and namespace to make sure the qualified table name is expected, and also check the catalog implementation which is configured by "spark.sql.catalog". SQLSTATE: 0A000